# POP77032 Assignment 1: Text Preparation

## Before Submission

-   Make sure that you can run all cells without errors
-   You can do it by clicking `Kernel`, `Restart & Run All` in the menu
    above
-   Make sure that you save the output by pressing Command+S / CTRL+S
-   Rename the file from `01_assignment.ipynb` to
    `01_lastname_firstname_studentnumber.ipynb`
-   Use Firefox browser for submitting your Jupyter notebook on
    Blackboard.

## Overview

In this assignment you will need to collect and prepare textual data for
analysis. As the data source we will debates in the Dáil Éireann (Irish
Parliament) for the first 2 months of 2025 (but in practice once you
implement a solution for those it should be relatively straightforward
to scale up).

There are 2 broad strategies that can be used to obtain Dáil debates:

1.  Use the [Oireachtas
    website](https://www.oireachtas.ie/en/debates/find/) to scrape the
    debates using R (e.g. `rvest`) or Python (e.g. `Beautiful Soup`).
    There can be different strategies to solve this, but, crucially, the
    website is largely static, so dealing with it as a set of HTML files
    is quite manageable.
2.  Use the [Oireachtas API](https://api.oireachtas.ie/) to scrape the
    debates using R (e.g. `httr2`) or Python (e.g. `requests`). This
    might be a more advanced option, but it is also a lot more powerful
    and flexible. Importantly, this API does not require authentication,
    which makes working with it quite a bit simpler than with many other
    APIs.

## Part 1: Data Acquisition

In this part you will need to write a scraper that collects the data
either directly from the Oireachtas website or using the Oireachtas API.
The data should be collected for the first 2 months of 2025 (January and
February, but the bulk of the debates would be in February).

Depending on how you choose to organise your code, you may choose to
build up a usual tabular dataset straightaway or you might find it
easier to store the data in a different container (e.g. a list of
vectors, a list of lists, a list of dictionaries, etc.) and then convert
it to a tabular format in the next part.

You may use generative AI to help you with trialing different
approaches. If you do use AI, you need to report the version of the LLM
that you are using (e.g. `code-davinci-002`,
`meta-llama-3.1-8b-intruct`, etc.). Hardware permitting, I encourage you
to use offline models to have better control over the data and the
model.

While there maybe also some bindings for the API that are readily
available, none of them are officially supported, so you shouldn’t be
relying on those.

## Part 2: Text Preprocessing

In this part you will need to clean up the collected data. Depending on
how the previous part was implemented it might take more or fewer steps.
The ultimate goal is to have a dataset of the following form:

| dail | vol | no  | date | speaker | text | ntokens | ntypes |
|------|-----|-----|------|---------|------|---------|--------|

where:

`dail` - is the number of the Dáil (e.g. 34th Dáil)

`vol` - is the volume number of the debates (e.g. 1000)

`no` - is the number of the debate in the volume (e.g. 1)

`date` - is the date of the debate (in YYYY-MM-DD form, e.g. 2025-01-01)

`speaker` - is the name of the speaker

`text` - is the text of the speech

`ntokens` - is the number of tokens in the speech

`ntypes` - is the number of types in the speech

Note that you **do not** need to submit the actual dataset. However,
after organising the textual data in this way, you will need to perform
the following steps:

-   Print out the first and last 5 rows of the data
-   Print the dimensionality of the data (number of rows and number of
    columns)
-   Print the total number of unique speakers in the dataset.
-   Print the average number of tokens per speech.
-   Print the average number of types per speech.

## Part One - Data Acquisition 
 In this section, I first scrape the main debate pages and then scrape each individual debate within those pages. 

In [4]:
#Libraries
libs <- c("tidyverse",
          "rvest",
          "httr",
          "stringr",
          "quanteda")
install.packages(setdiff(libs, rownames(installed.packages())))

lapply(libs, library, character.only = TRUE)

#using a link from the website directly filtered by date
url <- "https://www.oireachtas.ie/en/debates/find/?debateType=dail&datePeriod=dates&fromDate=01%2F01%2F2025&toDate=27%2F02%2F2025&term=%2Fie%2Foireachtas%2Fhouse%2Fdail%2F34&committee=&member="

#Extracting the links of each main debate page
extract_links <- function(page_url) {
  page <- read_html(page_url)
  hrefs <- page %>%
    html_nodes("a.u-btn-secondary.c-debates-expanding-list__button") %>%
    html_attr("href")
  hrefs_absoloute <- url_absolute(hrefs, page_url)
  return(hrefs_absoloute)
}

debate_links <- url %>% map(extract_links) %>% unlist()
debate_links[1]
pages <- vector("list", length(debate_links))

#after extracting the links, this loop will read each page
for (i in seq_along(debate_links)) {
  message("Reading: ", debate_links[i])
  pages[[i]] <- read_html(debate_links[i])
}

#Because the daily is not on each debate, I will subset it from the main pages
#Following this, I will examine which Dail it is and if it is a constant for my dataframe later

dail <- vector("character", length(pages))

for (i in seq_along(pages)) {
  dail[i] <- pages[[i]] %>%
    html_nodes("p.c-debate-header__meta-item:nth-of-type(2)") %>%  
    html_text(trim = TRUE)
}

dail[1] 

#This function will be used to extract each debate within the main page.
extract_links <- function(page_url) {
  page <- read_html(page_url)
  hrefs <- page %>%
    html_nodes("a") %>%
    html_attr("href") %>%
    .[grepl(
      "/en/debates/debate/dail/\\d{4}-\\d{2}-\\d{2}/\\d+/(#s\\d+)?$", #using regex to represent all the various endings to the url each individual debate has
      .
    )]
  hrefs_absoloute <- url_absolute(hrefs, page_url)
  return(hrefs_absoloute)
}

debate <- debate_links %>% map(extract_links) %>% unlist() #running the function

# Reading each debate within the main debate pages after we've extracted it.
debates <- vector("list", length(debate))
for (i in seq_along(debate)) {
  message("Reading: ", debate[i])
  debates[[i]] <- read_html(debate[i])
}

[[1]]
 [1] "quanteda"  "httr"      "rvest"     "lubridate" "forcats"   "stringr"  
 [7] "dplyr"     "purrr"     "readr"     "tidyr"     "tibble"    "ggplot2"  
[13] "tidyverse" "repr"      "stats"     "graphics"  "grDevices" "utils"    
[19] "datasets"  "methods"   "base"     

[[2]]
 [1] "quanteda"  "httr"      "rvest"     "lubridate" "forcats"   "stringr"  
 [7] "dplyr"     "purrr"     "readr"     "tidyr"     "tibble"    "ggplot2"  
[13] "tidyverse" "repr"      "stats"     "graphics"  "grDevices" "utils"    
[19] "datasets"  "methods"   "base"     

[[3]]
 [1] "quanteda"  "httr"      "rvest"     "lubridate" "forcats"   "stringr"  
 [7] "dplyr"     "purrr"     "readr"     "tidyr"     "tibble"    "ggplot2"  
[13] "tidyverse" "repr"      "stats"     "graphics"  "grDevices" "utils"    
[19] "datasets"  "methods"   "base"     

[[4]]
 [1] "quanteda"  "httr"      "rvest"     "lubridate" "forcats"   "stringr"  
 [7] "dplyr"     "purrr"     "readr"     "tidyr"     "tibble"    "ggplot2"  
[13] "tidyverse" "repr"      "stats"     "graphics"  "grDevices" "utils"    
[19] "datasets"  "methods"   "base"     

[[5]]
 [1] "quanteda"  "httr"      "rvest"     "lubridate" "forcats"   "stringr"  
 [7] "dplyr"     "purrr"     "readr"     "tidyr"     "tibble"    "ggplot2"  
[13] "tidyverse" "repr"      "stats"     "graphics"  "grDevices" "utils"    
[19] "datasets"  "methods"   "base"

[1] "https://www.oireachtas.ie/en/debates/debate/dail/2025-02-27/"

Reading: https://www.oireachtas.ie/en/debates/debate/dail/2025-02-27/

Reading: https://www.oireachtas.ie/en/debates/debate/dail/2025-02-26/

Reading: https://www.oireachtas.ie/en/debates/debate/dail/2025-02-25/

Reading: https://www.oireachtas.ie/en/debates/debate/dail/2025-02-20/

Reading: https://www.oireachtas.ie/en/debates/debate/dail/2025-02-19/

Reading: https://www.oireachtas.ie/en/debates/debate/dail/2025-02-18/

Reading: https://www.oireachtas.ie/en/debates/debate/dail/2025-02-13/

Reading: https://www.oireachtas.ie/en/debates/debate/dail/2025-02-12/

Reading: https://www.oireachtas.ie/en/debates/debate/dail/2025-02-11/

Reading: https://www.oireachtas.ie/en/debates/debate/dail/2025-02-06/

Reading: https://www.oireachtas.ie/en/debates/debate/dail/2025-02-05/

Reading: https://www.oireachtas.ie/en/debates/debate/dail/2025-01-23/

Reading: https://www.oireachtas.ie/en/debates/debate/dail/2025-01-22/



[1] "34th Dáil"

Reading: https://www.oireachtas.ie/en/debates/debate/dail/2025-02-27/1/

Reading: https://www.oireachtas.ie/en/debates/debate/dail/2025-02-27/2/

Reading: https://www.oireachtas.ie/en/debates/debate/dail/2025-02-27/2/#s3

Reading: https://www.oireachtas.ie/en/debates/debate/dail/2025-02-27/2/#s4

Reading: https://www.oireachtas.ie/en/debates/debate/dail/2025-02-27/2/#s5

Reading: https://www.oireachtas.ie/en/debates/debate/dail/2025-02-27/2/#s6

Reading: https://www.oireachtas.ie/en/debates/debate/dail/2025-02-27/2/#s7

Reading: https://www.oireachtas.ie/en/debates/debate/dail/2025-02-27/8/

Reading: https://www.oireachtas.ie/en/debates/debate/dail/2025-02-27/8/#s9

Reading: https://www.oireachtas.ie/en/debates/debate/dail/2025-02-27/8/#s10

Reading: https://www.oireachtas.ie/en/debates/debate/dail/2025-02-27/8/#s11

Reading: https://www.oireachtas.ie/en/debates/debate/dail/2025-02-27/8/#s12

Reading: https://www.oireachtas.ie/en/debates/debate/dail/2025-02-27/8/#s13

Reading: https://

## Part Two

 In this section, I am continuing to scrape; however I am creating vectors with the variables of interest as I go and will compile them into a dataframe at the very end. 

In [17]:
#extracting the date from each individual debate
date <- vector("character", length(debates))
for (i in seq_along(debates)) {
  date[i] <- debates[[i]] %>%
    html_nodes("h1.c-hero__title") %>%
    html_text(trim = TRUE) %>%
    str_remove(".*?,\\s*") %>%
    str_replace("(\\d{1,2}) ([A-Za-z]{3}) (\\d{4})", "\\3-\\2-\\1") %>% #using regex to remove the beginning of the date string and then rearranging the order
    str_replace_all(c("Jan"= "01", "Feb"= "02"))
}

date[1]

vol <- vector("character", length(pages)) #Extracting both Vol and No since they are printed together

for (i in seq_along(debates)) {
  vol[i] <- debates[[i]] %>%
    html_nodes("p.c-hero__subtitle") %>%
    html_text(trim = TRUE)
  
}

# Since they are printed together, I have to use regex to select both of them individually. 
vols <- str_extract(vol, "(?<=Vol\\.\\s)\\d+")
nums <- str_extract(vol, "(?<=No\\.\\s)\\d+")

vols[1]
nums[1]

#I noticed that each speaker is boldened, so to obtain each speaker, I selected the "strong" text in each debate.
speakers <- vector("character", length(debates))
for (i in seq_along(debates)) {
  speakers[i] <- debates[[i]] %>%
    html_nodes("strong") %>%
    html_text(trim = TRUE) %>%
    unique() %>%
    paste(collapse = "; ")
}

speakers[2]

#Extracting all text from each indivdual debate
texts <- vector("character", length(debates))
for (i in seq_along(debates)) {
  texts[i] <- debates[[i]] %>%
    html_nodes("p[id^='para_']") %>%
    html_text(trim = TRUE) %>%
    paste(collapse = " ")
}

texts[2]

ntokens <- integer(length(texts)) 
ntypes <- integer(length(texts))
#extracting tokens and types
for (i in seq_along(texts)) {
  clean_text <- texts[[i]] 
  clean_text <- tolower(clean_text) 
  clean_text <- str_remove_all(clean_text, "[[:punct]]")  #doing some very light cleaning by removing punctuation, given that our objective is unknown.
  clean_text <- str_remove_all(clean_text,"\\d+") 
  
  token <- quanteda::tokens(clean_text) 
  
  ntokens[i] <- quanteda::ntoken(token) #counting all tokens
  ntypes[i] <- quanteda::ntype(token) #counting unique tokens
}

[1] "2025-02-27"

[1] "1063"

[1] "6"

[1] "Deputy Darren O'Rourke; Minister for Education; Deputy Ruth Coppinger; Deputy Richard O'Donoghue"

[1] "1. Deputy Darren O'Rourke asked the Minister for Education if she is aware of the concerns raised by school leaders in relation to the accelerated roll-out of senior cycle reform; if she accepts that the accelerated timeline does not allow enough time for essential preparations; and if she will make a statement on the matter. [8924/25] As this is the first ministerial questions session, I wish the Minister and Minister of State the best of luck in their roles. This is a very important time and I hope we can make progress. Is the Minister aware of the concerns raised by school leaders with regard to the accelerated roll-out of senior cycle reform, if she accepts that the accelerated timeline does not allow enough time for essential preparations and if she will make a statement on the matter? I look forward to working with all colleagues, particularly in this area. I thank the Deputy for asking this question as it is an important and live issue, in particular for students who will sit the leaving certificate in the years ahead. It is for this reason that, in the last few weeks, I have met nearly all the unions and representative bodies representing teachers, principals, schools in general, boards of management, parents and, importantly, students. These conversations were not just about senior cycle reform. What was clear when it came to senior cycle reform is that we are all on the same page and want to ensure we have a senior cycle that is modern, fit for purpose and, above all, prepares our young people for the world we live in today. What is also clear and what we all agree on is that we need to support our teachers and schools in rolling out that programme. I am absolutely committed to continuing the programme of reform of senior cycle to ensure that students benefit from up-to-date curriculums, more diverse skills development and assessments and that we also reduce stress levels. I am also absolutely committed to working with schools, school leaders and teachers in that regard. The world is changing rapidly and it is essential for all of us to properly equip students to succeed in this changing world. The approach to senior cycle redevelopment is about collaboration and engagement with our stakeholders and partners in education to try to deliver this. That engagement commenced back in 2016 and there has been detailed engagement and consultation since then. In 2023, it was announced that the first tranche of two new subjects and seven revised subjects would be introduced in schools in September of this year. Therefore, the first class taking these new and updated subjects will complete their leaving certificate in 2027. The introduction of the new assessment models will help students to develop new skills and reduce the amount of pressure on students. This builds on what is already in place across 28 of the 40 curricular leaving certificate subjects. This is not new when it comes to the additional assessment component but we are trying to modernise and ensure that it applies to the vast majority of subjects. I welcome the fact that the Minister has met stakeholders. It is important that she meets all of the stakeholders in the sector at the earliest opportunity. If she has met them, she must surely have heard their concerns. I have not heard that reflected in her response thus far but they will have detailed them to her. They relate to a range of areas, principally capacity and preparedness, whether in terms of the development of curriculums, the readiness of schools in terms of IT, laboratories and the capacity to take on these changes. The Minister said there is a commitment, as I believe there is, to make this happen in the right way but that takes time and capacity. Is she hearing that from the stakeholders and, if so, how is she responding? I think our stakeholders have outlined a number of things clearly. First, they want to ensure teachers feel appropriately trained and equipped because they want to ensure they a

In [9]:
dail_all <- rep("34th Dáil", 299) 
#since there is no Dail printed for each debate and they are the same one,
#I just extended the Dail to the length of debates.

#creating the dataframe
dataf <- data.frame(
  Dail = dail_all,
  Vol = vols,
  No = nums,
  Date = date,
  Speakers = speakers,
  Text = texts,
  Ntoken = ntokens,
  Ntypes = ntypes,
  stringsAsFactors = FALSE
)

glimpse(dataf)

Rows: 299
Columns: 8
$ Dail     <chr> "34th Dáil", "34th Dáil", "34th Dáil", "34th Dáil", "34th Dái…
$ Vol      <chr> "1063", "1063", "1063", "1063", "1063", "1063", "1063", "1063…
$ No       <chr> "6", "6", "6", "6", "6", "6", "6", "6", "6", "6", "6", "6", "…
$ Date     <chr> "2025-02-27", "2025-02-27", "2025-02-27", "2025-02-27", "2025…
$ Speakers <chr> "", "Deputy Darren O'Rourke; Minister for Education; Deputy R…
$ Text     <chr> "", "1. Deputy Darren O'Rourke asked the Minister for Educati…
$ Ntoken   <int> 0, 6404, 6404, 6404, 6404, 6404, 6404, 10143, 10143, 10143, 1…
$ Ntypes   <int> 0, 1018, 1018, 1018, 1018, 1018, 1018, 1417, 1417, 1417, 1417…


## Part Three - Responses to Statements

In [10]:
#Printing First Five Rows
head(dataf,5)

Dail      Vol  No Date      
1 34th Dáil 1063 6  2025-02-27
2 34th Dáil 1063 6  2025-02-27
3 34th Dáil 1063 6  2025-02-27
4 34th Dáil 1063 6  2025-02-27
5 34th Dáil 1063 6  2025-02-27
  Speakers                                                                                        
1                                                                                                 
2 Deputy Darren O'Rourke; Minister for Education; Deputy Ruth Coppinger; Deputy Richard O'Donoghue
3 Deputy Darren O'Rourke; Minister for Education; Deputy Ruth Coppinger; Deputy Richard O'Donoghue
4 Deputy Darren O'Rourke; Minister for Education; Deputy Ruth Coppinger; Deputy Richard O'Donoghue
5 Deputy Darren O'Rourke; Minister for Education; Deputy Ruth Coppinger; Deputy Richard O'Donoghue
  Text                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                

In [11]:
#Printing Last Five Rows
tail(dataf,5)

Dail      Vol  No Date       Speakers
295 34th Dáil 1062 3  2025-01-23         
296 34th Dáil 1062 3  2025-01-23         
297 34th Dáil 1062 2  2025-01-22         
298 34th Dáil 1062 2  2025-01-22         
299 34th Dáil 1062 2  2025-01-22         
    Text                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                

In [12]:
#Printing the Dimensionality of the data
dim(dataf)

[1] 299   8

In [13]:
#Print total number of unique speakers in the dataset
all_speakers <- unlist(strsplit(speakers, ";")) #unlisting and sep all speakers
total_speakers <- length(unique(all_speakers))
total_speakers

[1] 94

In [15]:
#Mean Tokens after doing some light scraping.
mean(ntokens)

[1] 8029.452

In [14]:
#mean types after doing some light scraping.
mean(ntypes)

[1] 1273.572